# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Mã nằm trong package `day10/` (chạy được cả trên Colab T4 lẫn máy thuê Vast.ai); notebook gọi lại từng bước.
Muốn chạy hết một lèo trên terminal: `SMOKE=1 bash run_all.sh` (thử ~10 phút) rồi `bash run_all.sh`.

| Bước | Module |
|---|---|
| nhãn, prompt | `day10/common.py` |
| encode + mask loss + collate | `day10/data.py` |
| nạp 4-bit, LoRA E1/E2/E3, projector | `day10/modeling.py` |
| huấn luyện / suy luận / chấm | `day10/train.py`, `day10/infer.py`, `day10/evidence.py` |
| GRPO + reward | `day10/grpo.py`, `day10/reward.py` (test: `day10/tests/`) |
| phân tích, ảnh | `day10/analyze.py`, `day10/coord_check.py` |

Mỗi lần huấn luyện/suy luận chạy trong **tiến trình con** (`!python -m ...`) → GPU được giải phóng sau mỗi bước, không phải restart runtime giữa E1/E2/E3.

## 0. Môi trường (Colab T4 hoặc Vast.ai)

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/tam253211-a11y/K4-Track-4-Day10-DangHuuTam-2A202602940-VisionLanguageModel.git"
if not os.path.exists("scripts/score.py"):               # chưa đứng trong repo → clone rồi cd vào
    REPO_DIR = os.path.join("/content" if IN_COLAB else os.getcwd(), REPO_URL.rsplit("/", 1)[-1][:-4])
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
os.environ["PYTHONPATH"] = os.getcwd()
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, os.getcwd())
!nvidia-smi -L
!pip -q install -r requirements-colab.txt pytest matplotlib datasets requests

In [ ]:
# Công cụ của đề: kỳ vọng 68 passed, 1 skipped (test dữ liệu thật skip khi chưa có data/)
!python -m pytest -q tests
# Test của mình (nhãn ↔ trọng tài, reward E5) — không cần GPU
!python -m pytest -q day10/tests

In [ ]:
# Sao lưu adapter (Colab free có thể bị ngắt). Trên Vast: để BACKUP_DIR = "" (đĩa máy thuê không mất giữa chừng)
BACKUP_DIR = ""
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_DIR = "/content/drive/MyDrive/day10_vlm"
    os.makedirs(BACKUP_DIR, exist_ok=True)
BK = f"--backup-dir {BACKUP_DIR}" if BACKUP_DIR else ""

In [ ]:
# Kaggle: Colab → biểu tượng 🔑 → thêm KAGGLE_USERNAME, KAGGLE_KEY và BẬT "Notebook access".
# Vast: export KAGGLE_USERNAME=... KAGGLE_KEY=... trong terminal trước khi mở Jupyter. KHÔNG viết khoá vào notebook.
if IN_COLAB:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception as e:
            print(f"thiếu secret {k}: {type(e).__name__} — tạo secret và bật Notebook access, "
                  "hoặc dùng --zip ở cell dưới")
print({k: bool(os.environ.get(k)) for k in ("KAGGLE_USERNAME", "KAGGLE_KEY")})

## 1. Dữ liệu: tải và chia split theo phiên quay

In [ ]:
if not os.path.exists("data/splits/meta.json"):
    !python scripts/prepare_data.py --kaggle
    # Không có Kaggle key: tải zip tay, upload lên Drive rồi dùng:
    # !python scripts/prepare_data.py --zip "/content/drive/MyDrive/<file>.zip"

In [ ]:
# Kiểm tra: dataset_version d10-…, số ảnh/box đúng bảng đề, không session nào ở hai split; vẽ 6 ảnh có box
%matplotlib inline
from day10.check_data import check, figure
splits, meta = check()
figure(splits);

## 2. Nhãn: YOLO → JSON thang 0–1000
`day10/common.py`: `yolo_to_box` (tâm+kích thước [0,1] → góc ×1000, làm tròn, kẹp [0,1000], không để box suy biến vì
`score.parse_output` coi `x2<=x1` là JSON hỏng) và `objects_to_answer` (sắp theo `(y1, x1)`, JSON không khoảng trắng).

In [ ]:
import inspect, json
from day10 import common
print(inspect.getsource(common.yolo_to_box)); print(inspect.getsource(common.objects_to_answer))
print("PROMPT:", common.get_prompt("norm1000"))
rec = next(r for r in splits["train"] if len(r["objects"]) >= 3)
print("answer:", common.objects_to_answer(rec["objects"]))

In [ ]:
# Kiểm tra quy đổi ngược: box JSON 0–1000 → pixel, vẽ viền trắng chồng lên box gốc (màu) — phải trùng khít
import matplotlib.pyplot as plt
from PIL import ImageDraw
from day10.check_data import draw_gt
img = draw_gt(rec); d = ImageDraw.Draw(img); W, H = img.size
for b in json.loads(common.objects_to_answer(rec["objects"])):
    x1, y1, x2, y2 = b["bbox_2d"]
    d.rectangle([x1 * W / 1000, y1 * H / 1000, x2 * W / 1000, y2 * H / 1000], outline="white", width=1)
plt.figure(figsize=(10, 6)); plt.imshow(img); plt.axis("off"); plt.show()
# Câu 3: dùng chính trọng tài để thấy lệch thang làm F1 sụp dù JSON hợp lệ
!python -m day10.coord_check

## 3. E0: zero-shot

In [ ]:
# Model gốc 4-bit NF4, llm_int8_skip_modules=["visual","lm_head"] (ViT + projector + lm_head giữ fp16). Batch 4, greedy.
# Gỡ lỗi trước trên 8 ảnh val, xem output thô:
!python -m day10.infer --zero-shot --split val --limit 8 --out work/debug/e0_val8.jsonl

In [ ]:
!python -m day10.infer --zero-shot --out work/e0.jsonl
!python -m day10.evidence --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
!python -m day10.analyze e0 --pred work/e0.jsonl

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
`day10/data.py::encode_example`: chuỗi prompt (ảnh + câu lệnh, `add_generation_prompt=True`) phải là tiền tố của chuỗi
đầy đủ; `labels[:n_prompt] = -100` → chỉ học JSON đáp án + `<|im_end|>`. `collate` pad `input_ids/attention_mask/`
`mm_token_type_ids/labels` (labels đệm -100) và **nối** `pixel_values/image_grid_thw` theo chiều 0.
> transformers 5.19 bắt buộc có `mm_token_type_ids` (M-RoPE) — collate mẫu trong đề thiếu khoá này sẽ báo ValueError.

In [ ]:
import torch
from day10.data import GroundingDataset, make_collate, num_image_tokens
from day10.modeling import load_processor
processor = load_processor()
ds = GroundingDataset(splits["train"], processor)
ex = ds[0]
print("keys:", list(ex))
print("image_grid_thw:", ex["image_grid_thw"].tolist(), "→ token ảnh =", num_image_tokens(ex["image_grid_thw"]),
      "(Qwen3-VL patch 16, gộp 2×2: 640×360 → 640×352 → 40×22 patch → 20×11 = 220)")
n_learn = int((ex["labels"] != -100).sum()); n_all = ex["input_ids"].numel()
print(f"seq={n_all} token, học {n_learn} token ({n_learn / n_all:.1%}), mask {n_all - n_learn}")
print("labels != -100 giải mã:", repr(processor.tokenizer.decode(ex["input_ids"][ex["labels"] != -100])))
print("đáp án               :", repr(common.objects_to_answer(splits["train"][0]["objects"])))
batch = make_collate(processor.tokenizer.pad_token_id)([ds[0], ds[1]])
print({k: tuple(v.shape) for k, v in batch.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# day10/modeling.py::apply_setup — regex trên tên module thật (PEFT fullmatch), assert 252 (E1/E2) và 348 (E3).
# KHÔNG dùng list hậu tố ["qkv","linear_fc1","linear_fc2"]: sẽ khớp nhầm merger/deepstack (cũng có linear_fc1/2) và thiếu attn.proj.
from day10.modeling import LLM_RE, VIT_RE, architecture_breakdown
print("LLM:", LLM_RE); print("ViT:", VIT_RE)
arch = architecture_breakdown()      # đếm trên model 'meta' (không tải trọng số) — số cho Câu 1
for k, v in arch.items():
    print(f"{k:32s} {v:,}" if isinstance(v, int) and not isinstance(v, bool) else f"{k:32s} {v}")

## 6. Vòng huấn luyện
AdamW lr 2e-4 (wd 0), warmup 5% + cosine, GradScaler fp16, gradient checkpointing (use_reentrant=False), batch 2 × accum 4,
300 bước, clip 1.0. Lưu adapter **và** `projector.pt` (E2/E3) mỗi 100 bước; ghi `run.json` (trainable_params, peak_vram_gb,
train_minutes, loss). Hết VRAM: thêm `--batch-size 1 --grad-accum 8` (batch hiệu dụng vẫn 8, ghi chú trong báo cáo).

In [ ]:
# Chạy thử 3 bước để bắt lỗi sớm (xem: 220 token ảnh, số module LoRA, VRAM)
!python -m day10.train --config configs/e1.yaml --out work/smoke_e1 --max-steps 3

In [ ]:
!python -m day10.train --config configs/e1.yaml --out work/e1 {BK}

In [ ]:
!python -m day10.train --config configs/e2.yaml --out work/e2 {BK}

In [ ]:
!python -m day10.train --config configs/e3.yaml --out work/e3 {BK}

## 7. Suy luận và chấm E1–E3
Nạp model gốc 4-bit → adapter → projector (`load_projector` in checksum trước/sau để chắc projector đã đổi).
`day10.evidence` lấy `--meta` **nguyên văn** từ `run.json` (không gõ tay số). Mẹo: gỡ lỗi với `--split val --limit 40`.

In [ ]:
!python -m day10.infer --run work/e1 --split val --limit 40 --out work/debug/e1_val40.jsonl
!python -m day10.evidence --pred work/debug/e1_val40.jsonl --name E1val --split val --run work/e1 --out work/debug/e1_val40.json

In [ ]:
for k in ("e1", "e2", "e3"):
    !python -m day10.infer --run work/{k} --out work/{k}.jsonl
    !python -m day10.evidence --pred work/{k}.jsonl --name {k.upper()} --run work/{k} --out report/evidence/{k}.json
!python -m day10.analyze table

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
**E4 = data_size** trên setup tốt nhất của E1–E3: chỉ đổi số ảnh train (10% vs 100%), giữ nguyên mọi thứ khác
(kể cả 300 bước → 10% dữ liệu lặp ~17 epoch). Giá trị 100% chính là setup gốc (GUIDE cho phép) → chỉ chấm lại.

In [ ]:
f1 = {k: json.load(open(f"report/evidence/{k}.json"))["micro"]["f1"] for k in ("e1", "e2", "e3")}
BEST = max(f1, key=f1.get); print(f1, "→ setup gốc cho E4/E5:", BEST)
!python -m day10.evidence --pred work/{BEST}.jsonl --name E4_data_1.0 --run work/{BEST} --out report/evidence/e4_data_1.0.json
!python -m day10.train --config configs/{BEST}.yaml --out work/e4_data_0.1 --name E4_data_0.1 --train-fraction 0.1 {BK}
!python -m day10.infer --run work/e4_data_0.1 --out work/e4_data_0.1.jsonl
!python -m day10.evidence --pred work/e4_data_0.1.jsonl --name E4_data_0.1 --run work/e4_data_0.1 --out report/evidence/e4_data_0.1.json

In [ ]:
# E5 (tuỳ chọn): test reward trước, rồi GRPO 60 bước từ adapter tốt nhất
!python -m pytest -q day10/tests/test_reward.py
!python -m day10.grpo --init work/{BEST} --out work/e5 {BK}
!python -m day10.infer --run work/e5 --out work/e5.jsonl
!python -m day10.evidence --pred work/e5.jsonl --name E5 --run work/e5 --out report/evidence/e5.json
!python -m day10.analyze hacking --pred work/{BEST}.jsonl work/e5.jsonl

In [ ]:
# Phân tích cho REPORT.md: bảng, lỗi theo lớp/kích thước (Câu 9), ảnh minh hoạ (≥6, box dự đoán + box đúng)
!python -m day10.analyze table
!python -m day10.analyze classes --pred work/{BEST}.jsonl
!python -m day10.analyze figures --pred work/{BEST}.jsonl --tag {BEST}
!python -m day10.coord_check --pred work/{BEST}.jsonl
!mkdir -p report/runs && for d in work/e*/; do [ -f $d/run.json ] && cp $d/run.json report/runs/$(basename $d).json; done; cp work/*.info.json report/runs/ 2>/dev/null; true
from IPython.display import Image as IPImage, display
display(IPImage(f"report/figures/{BEST}_examples.png"))

In [ ]:
# Kiểm tra trước khi nộp (cần data/splits/meta.json để mục 'khớp meta.json' PASS)
!python scripts/check_submission.py